# GEC block assessment, aquifer zones of wells, and DEM

**Alwar groundwater study · 14-block domain (LGD 574–587)**

Three inputs for the model that come from data already on Drive:
1. **GEC 2024-25** recharge and extraction per block (the pumping input and the check on modelled recharge);
2. **aquifer formation** of the 474 monitoring wells, grouped into alluvium / alluvium over hard rock / hard rock;
3. the **Copernicus 30 m DEM** and slope (model top; hard-rock hills).

| | |
|---|---|
| **Reads** | `data/CGRA_Reports/stateReport 2024-25.xlsx`, `inputs/alwar/blocks.geojson` and `climate_obs/` (from the climate notebook), the cleaned well table, Step 17 block trends |
| **Writes** | `inputs/alwar/gec/`, `inputs/alwar/aquifer/`, `inputs/alwar/dem_alwar_utm43n_30m.tif`, `slope_alwar_deg.tif` |

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["geopandas", "rasterio", "openpyxl", "requests"], modflow=False)
del TOKEN, URL


## 1. GEC 2024-25 block assessment → clean table
The workbook `stateReport 2024-25.xlsx` has one sheet (`GEC`) with a three-row header (rows 7–9) and the
14 blocks in rows 11–24. This cell flattens the header to one name per column.

In [ ]:
# ── G1: flatten the 3-row GEC header → one clean table for the 14 blocks ──
import os
import numpy as np
import pandas as pd

GW = "/content/drive/MyDrive/gw_framework"
XL = f"{GW}/data/CGRA_Reports/stateReport 2024-25.xlsx"
OUT = f"{GW}/inputs/alwar/gec"
os.makedirs(OUT, exist_ok=True)
H1, H2, H3, D0, D1 = 7, 8, 9, 11, 24      # header rows and data rows (0-based)

raw = pd.read_excel(XL, sheet_name="GEC", header=None, dtype=object)

def clean(v):
    return "" if pd.isna(v) else " ".join(str(v).split())

# level 1 and 2 are merged cells → carry them right; level 2 resets when level 1 changes
names, l1, l2 = [], "", ""
for j in range(raw.shape[1]):
    a, b, c = clean(raw.iat[H1, j]), clean(raw.iat[H2, j]), clean(raw.iat[H3, j])
    if a:
        l1, l2 = a, ""
    if b:
        l2 = b
    parts = [p for p in (l1, l2 if l2 != l1 else "", c) if p]
    names.append(" | ".join(parts) if parts else f"col_{j}")

seen = {}                                   # make names unique
for k, n in enumerate(names):
    if n in seen:
        seen[n] += 1; names[k] = f"{n} #{seen[n]}"
    else:
        seen[n] = 0

df = raw.iloc[D0:D1 + 1].copy()
df.columns = names
df = df.loc[:, [c for c in df.columns if df[c].notna().any()]]     # drop empty columns
df.to_csv(f"{OUT}/gec_2024_25_alwar_flat.csv", index=False)
print(f"Blocks: {len(df)} | non-empty columns: {df.shape[1]}")
print("Blocks:", df["BLOCK"].tolist())
print("Saved:", f"{OUT}/gec_2024_25_alwar_flat.csv")

## 2. Key GEC quantities per block, in model units
ham → mm over the recharge-worthy area and m³/day (1 ham = 10,000 m³).
GEC areas differ from the GIS block areas (ratio 0.75–1.66), so apply GEC as **mm/yr rates**, not volumes.

In [ ]:
# ── G2: key GEC 2024-25 quantities per block → model units ──
import numpy as np
import pandas as pd
import geopandas as gpd

GW = "/content/drive/MyDrive/gw_framework"
ALW = f"{GW}/inputs/alwar"
flat = pd.read_csv(f"{ALW}/gec/gec_2024_25_alwar_flat.csv")

COLS = {
    "rainfall_mm_gec": "Rainfall (mm) | Total",
    "area_total_ha":   "Total Geographical Area (ha) | Total",
    "area_rw_ha":      "Total Geographical Area (ha) | Recharge Worthy Area (ha) | Total",
    "area_hilly_ha":   "Total Geographical Area (ha) | Hilly Area",
    "rech_rain_ham":   "Ground Water Recharge (ham) | Rainfall Recharge | Total",
    "rech_canal_ham":  "Ground Water Recharge (ham) | Canals | Total",
    "rech_swirr_ham":  "Ground Water Recharge (ham) | Surface Water Irrigation | Total",
    "rech_gwirr_ham":  "Ground Water Recharge (ham) | Ground Water Irrigation | Total",
    "rech_tanks_ham":  "Ground Water Recharge (ham) | Tanks and Ponds | Total",
    "rech_total_ham":  "Annual Ground water Recharge (ham) | Total",
    "env_flow_ham":    "Environmental Flows (ham) | Total",
    "extractable_ham": "Annual Extractable Ground water Resource (ham) | Total",
    "extr_dom_ham":    "Ground Water Extraction for all uses (ha.m) | Domestic | Total",
    "extr_ind_ham":    "Ground Water Extraction for all uses (ha.m) | Industrial | Total",
    "extr_irr_ham":    "Ground Water Extraction for all uses (ha.m) | Irrigation | Total",
    "extr_total_ham":  "Ground Water Extraction for all uses (ha.m) | Total",
    "stage_pct":       "Stage of Ground Water Extraction (%) | Total",
    "category":        "Categorization of Assessment Unit | Total",
    "trend_pre":       "Pre Monsoon of GW Trend | NC",
    "trend_post":      "Post Monsoon of GW Trend | NC",
}
missing = [v for v in COLS.values() if v not in flat.columns]
assert not missing, f"columns not found: {missing}"

g = flat[["BLOCK"] + list(COLS.values())].copy()
g.columns = ["BLOCK"] + list(COLS.keys())
g["BLOCK"] = g["BLOCK"].str.upper().str.strip()

blocks = gpd.read_file(f"{ALW}/blocks.geojson")
blocks["BLOCK"] = blocks["block_name"].str.upper().str.strip()
blocks["bname"] = blocks["block_lgd"].astype(int).astype(str) + "_" + blocks["block_name"]
blocks["gis_area_ha"] = blocks.to_crs("EPSG:32643").area / 1e4
keep = blocks[["block_lgd", "bname", "BLOCK", "gis_area_ha"]]
g = keep.merge(g, on="BLOCK", how="outer", indicator=True)
print("Name match:", g["_merge"].value_counts().to_dict())
g = g.drop(columns="_merge")

# 1 ham = 10,000 m3 ; ham / ha = metres
ham_cols = [c for c in g.columns if c.endswith("_ham")]
for c in ham_cols:
    base = c[:-4]
    g[base + "_mm_rw"] = 1000 * g[c] / g["area_rw_ha"]
    g[base + "_m3d"] = g[c] * 1e4 / 365.0

g["rain_recharge_factor"] = g["rech_rain_mm_rw"] / g["rainfall_mm_gec"]
g["irr_return_fraction"] = g["rech_gwirr_ham"] / g["extr_irr_ham"]
g["net_deficit_ham"] = g["extr_total_ham"] - g["extractable_ham"]
g["net_deficit_mm_rw"] = 1000 * g["net_deficit_ham"] / g["area_rw_ha"]

imd = pd.read_csv(f"{ALW}/climate_obs/IMD_rain_AlwarBlocks.csv")
bcols = [c for c in imd.columns if c[:3].isdigit()]
ann = imd.groupby("Year")[bcols].sum()
g["imd_rain_2015_2024_mm"] = g["bname"].map(ann.loc[2015:2024].mean())
g["imd_rain_1985_2014_mm"] = g["bname"].map(ann.loc[1985:2014].mean())

g = g.sort_values("block_lgd").reset_index(drop=True)
g.to_csv(f"{ALW}/gec/gec_2024_25_blocks.csv", index=False)

pd.set_option("display.width", 230)
show = ["bname", "category", "stage_pct", "rainfall_mm_gec", "rech_total_mm_rw",
        "rain_recharge_factor", "extr_total_mm_rw", "irr_return_fraction", "net_deficit_mm_rw"]
print(g[show].round(2).to_string(index=False))
tot = g[ham_cols].sum()
print(f"\n14 blocks: recharge {tot['rech_total_ham']:,.0f} ham | extraction {tot['extr_total_ham']:,.0f} ham | "
      f"stage {100 * tot['extr_total_ham'] / tot['extractable_ham']:.0f}%")
print("Saved:", f"{ALW}/gec/gec_2024_25_blocks.csv")

## 3. Cross-check: specific yield implied by GEC deficit and observed decline
Sy ≈ (extraction − recharge, m/yr over recharge-worthy area) ÷ (median pre-monsoon Sen slope, m/yr).
Result: 0.015–0.09 for most blocks; not usable for Umren, Thanagazi, Laxmangarh and Kishangarh Bas.

In [ ]:
# ── G3: implied specific yield = GEC net deficit / observed decline ──
import numpy as np
import pandas as pd

GW = "/content/drive/MyDrive/gw_framework"
ALW = f"{GW}/inputs/alwar"
S17 = (f"{GW}/data/gwl/diagnostics/Alwar_Source_Recovery/"
       "Step_17_Block_Level_Trend_Assessment/03_step17_block_season_summary.csv")

gec = pd.read_csv(f"{ALW}/gec/gec_2024_25_blocks.csv")
tr = pd.read_csv(S17)
tr["BLOCK"] = tr["Block"].astype(str).str.upper().str.strip().replace({"UMRAIN": "UMREN"})
tr["Season"] = tr["Season"].astype(str).str.lower()

def season(s):
    t = tr[tr["Season"].str.startswith(s)][["BLOCK", "n_series_with_slope",
                                            "median_sen_slope_m_per_year",
                                            "sen_slope_q25_m_per_year", "sen_slope_q75_m_per_year"]]
    return t.add_prefix(s + "_").rename(columns={s + "_BLOCK": "BLOCK"})

d = gec.merge(season("pre"), on="BLOCK", how="left").merge(season("post"), on="BLOCK", how="left")
d["deficit_m"] = (d["extr_total_ham"] - d["rech_total_ham"]) / d["area_rw_ha"]
slope = d["pre_median_sen_slope_m_per_year"]            # + = water table deepening
d["Sy_implied_pre"] = np.where((slope > 0) & (d["deficit_m"] > 0), d["deficit_m"] / slope, np.nan)
q75, q25 = d["pre_sen_slope_q75_m_per_year"], d["pre_sen_slope_q25_m_per_year"]
d["Sy_low"] = np.where(q75 > 0, d["deficit_m"] / q75, np.nan)   # faster decline → lower Sy
d["Sy_high"] = np.where(q25 > 0, d["deficit_m"] / q25, np.nan)
d["gec_to_gis_area"] = d["area_total_ha"] / d["gis_area_ha"]

pd.set_option("display.width", 220)
cols = ["bname", "rain_recharge_factor", "deficit_m", "pre_n_series_with_slope",
        "pre_median_sen_slope_m_per_year", "Sy_implied_pre", "Sy_low", "Sy_high", "gec_to_gis_area"]
print(d[cols].round(3).to_string(index=False))
d.to_csv(f"{ALW}/gec/gec_vs_trends_implied_Sy.csv", index=False)

## 4. Aquifer formation of the monitoring wells
`Hyd_Formation` from the groundwater-department well table, grouped into three model classes:
alluvium, alluvium over hard rock, hard rock (with bedrock type).

In [ ]:
# ── G4: formation of domain wells (from Hyd_Formation) ──
import os
import pandas as pd
import geopandas as gpd

GW = "/content/drive/MyDrive/gw_framework"
ALW = f"{GW}/inputs/alwar"
OUT = f"{ALW}/aquifer"; os.makedirs(OUT, exist_ok=True)
WIDE = (f"{GW}/data/Rajasthan_gw_data/Groundwater_Cleaning_According_to_User_Rules/"
        "groundwater_cleaned_wide_source_preserved.csv")

w = pd.read_csv(WIDE, low_memory=False)
w["Latitude"] = pd.to_numeric(w["Latitude"], errors="coerce")
w["Longitude"] = pd.to_numeric(w["Longitude"], errors="coerce")
w["Total_Depth_bgl"] = pd.to_numeric(w["Total_Depth_bgl"], errors="coerce")
w = w.dropna(subset=["Latitude", "Longitude"])
pts = gpd.GeoDataFrame(w, geometry=gpd.points_from_xy(w.Longitude, w.Latitude), crs="EPSG:4326")

blocks = gpd.read_file(f"{ALW}/blocks.geojson")
blocks["bname"] = blocks.block_lgd.astype(int).astype(str) + "_" + blocks.block_name
inside = gpd.sjoin(pts, blocks[["bname", "geometry"]], how="inner", predicate="within")
inside["formation_raw"] = inside["Hyd_Formation"].fillna("(blank)").astype(str).str.strip()
print(f"Wells with coordinates: {len(pts):,} | inside 14-block domain: {len(inside):,}")
print(inside.formation_raw.value_counts().to_string())

keep = ["bname", "District", "Block", "Village ", "Well_Type", "Hyd_Formation", "formation_raw",
        "Latitude", "Longitude", "RL (m)", "Total_Depth_bgl"]
inside[[c for c in keep if c in inside.columns]].to_csv(f"{OUT}/wells_formation.csv", index=False)
print("Saved:", f"{OUT}/wells_formation.csv")

In [ ]:
# ── G5: group Hyd_Formation into model classes ──
import re
import pandas as pd

GW = "/content/drive/MyDrive/gw_framework"
AQ = f"{GW}/inputs/alwar/aquifer"
w = pd.read_csv(f"{AQ}/wells_formation.csv")

def rock_type(tok):
    t = tok.lower()
    if re.search(r"lime|lst|l\.st|dol", t):    return "carbonate"
    if re.search(r"gneiss|granite|pegmat", t): return "crystalline"
    if re.search(r"slate|phyl|schist|sch|^sl$|shale", t): return "metasediment"
    if re.search(r"quartz|qtz|^q$", t):        return "quartzite"
    return "other"

def classify(name):
    s = str(name).strip()
    low = s.lower().replace(" ", "")
    if low in ("alluvium", "all/boulder"):
        return "alluvium", "alluvium", ""
    if low.startswith("all"):                   # All/..., Alluvium/..., AllQtz
        rest = re.sub(r"^(alluvium|all)/?", "", low)
        return "alluvium_over_hardrock", "alluvium", rock_type(rest.split("/")[0] or rest)
    first = re.split(r"[/\-]", low)[0]
    return "hardrock", rock_type(first), ""

w[["zone_class", "surface_material", "basement_rock"]] = w["formation_raw"].apply(
    lambda x: pd.Series(classify(x)))
w.loc[w.zone_class == "hardrock", "basement_rock"] = w.loc[w.zone_class == "hardrock", "surface_material"]

pd.set_option("display.width", 200)
print("=== Mapping used (check this) ===")
print(w.groupby(["formation_raw", "zone_class", "basement_rock"]).size()
      .rename("wells").reset_index().sort_values(["zone_class", "wells"], ascending=[True, False])
      .to_string(index=False))
ct = pd.crosstab(w.bname, w.zone_class, margins=True)
ct["hardrock_share"] = (ct.get("hardrock", 0) / ct["All"]).round(2)
print("\n=== Wells per block by class ==="); print(ct.to_string())
w.to_csv(f"{AQ}/wells_formation_classified.csv", index=False)
print("Saved:", f"{AQ}/wells_formation_classified.csv")

## 5. Copernicus GLO-30 DEM → Alwar (UTM 43N, 30 m) and slope
Four tiles from the open AWS bucket (no login). Used for the model top and to map hard-rock hills.

In [ ]:
# ── G6: Copernicus GLO-30 DEM → Alwar (UTM 43N, 30 m) + slope + hilly fraction ──
import os, requests
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling

GW = "/content/drive/MyDrive/gw_framework"
ALW = f"{GW}/inputs/alwar"
RAW = f"{GW}/data/dem_copernicus"; os.makedirs(RAW, exist_ok=True)
DEM_UTM = f"{ALW}/dem_alwar_utm43n_30m.tif"
SLOPE = f"{ALW}/slope_alwar_deg.tif"
HILL_DEG = 8.0                         # slope threshold for "hilly" (degrees)

# 1. download tiles covering 27–29 N, 76–78 E
tiles = []
for lat in (27, 28):
    for lon in (76, 77):
        name = f"Copernicus_DSM_COG_10_N{lat:02d}_00_E{lon:03d}_00_DEM"
        url = f"https://copernicus-dem-30m.s3.amazonaws.com/{name}/{name}.tif"
        dst = f"{RAW}/{name}.tif"
        if not os.path.isfile(dst):
            r = requests.get(url, stream=True, timeout=300); r.raise_for_status()
            with open(dst + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 20): f.write(chunk)
            os.replace(dst + ".part", dst)
        tiles.append(dst); print("tile ok:", name, f"{os.path.getsize(dst)/1e6:.0f} MB")

# 2. mosaic and clip to domain + 5 km
domain = gpd.read_file(f"{ALW}/domain.geojson")
clip_geom = domain.to_crs("EPSG:32643").buffer(5000).to_crs("EPSG:4326")
srcs = [rasterio.open(t) for t in tiles]
mos, tr = merge(srcs); meta = srcs[0].meta.copy()
meta.update(height=mos.shape[1], width=mos.shape[2], transform=tr)
with rasterio.MemoryFile() as mf:
    with mf.open(**meta) as tmp:
        tmp.write(mos)
        clipped, ctr = mask(tmp, clip_geom.geometry, crop=True, nodata=-9999)
        cmeta = tmp.meta.copy()
cmeta.update(height=clipped.shape[1], width=clipped.shape[2], transform=ctr, nodata=-9999)

# 3. reproject to UTM 43N at 30 m
t2, w2, h2 = calculate_default_transform(cmeta["crs"], "EPSG:32643", cmeta["width"], cmeta["height"],
                                         *rasterio.transform.array_bounds(cmeta["height"], cmeta["width"], ctr),
                                         resolution=30)
dem = np.full((h2, w2), -9999, dtype="float32")
reproject(clipped[0], dem, src_transform=ctr, src_crs=cmeta["crs"], src_nodata=-9999,
          dst_transform=t2, dst_crs="EPSG:32643", dst_nodata=-9999, resampling=Resampling.bilinear)
umeta = dict(driver="GTiff", height=h2, width=w2, count=1, dtype="float32", crs="EPSG:32643",
             transform=t2, nodata=-9999, compress="deflate")
with rasterio.open(DEM_UTM, "w", **umeta) as d: d.write(dem, 1)

# 4. slope (degrees)
z = np.where(dem == -9999, np.nan, dem)
gy, gx = np.gradient(z, 30, 30)
slope = np.degrees(np.arctan(np.hypot(gx, gy))).astype("float32")
with rasterio.open(SLOPE, "w", **umeta) as d: d.write(np.nan_to_num(slope, nan=-9999), 1)
print(f"\nDEM: {w2} x {h2} cells | elevation {np.nanmin(z):.0f}–{np.nanmax(z):.0f} m | saved {DEM_UTM}")

# 5. hilly fraction per block vs GEC hilly area
blocks = gpd.read_file(f"{ALW}/blocks.geojson").to_crs("EPSG:32643")
blocks["bname"] = blocks.block_lgd.astype(int).astype(str) + "_" + blocks.block_name
gec = pd.read_csv(f"{ALW}/gec/gec_2024_25_blocks.csv").set_index("bname")
rows = []
with rasterio.open(SLOPE) as s, rasterio.open(DEM_UTM) as d:
    for _, b in blocks.iterrows():
        a, _ = mask(s, [b.geometry], crop=True, nodata=-9999)
        v = a[0][a[0] != -9999]
        e, _ = mask(d, [b.geometry], crop=True, nodata=-9999)
        ev = e[0][e[0] != -9999]
        rows.append(dict(bname=b.bname, elev_min=round(ev.min()), elev_median=round(np.median(ev)),
                         elev_max=round(ev.max()), slope_median=round(np.median(v), 1),
                         hilly_frac_dem=round((v > HILL_DEG).mean(), 3),
                         hilly_frac_gec=round(gec.loc[b.bname, "area_hilly_ha"] / gec.loc[b.bname, "area_total_ha"], 3)))
hf = pd.DataFrame(rows)
hf.to_csv(f"{ALW}/aquifer/block_terrain_summary.csv", index=False)
pd.set_option("display.width", 200)
print(hf.to_string(index=False))